In [1]:
import sys
from pathlib import Path

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.data.download_data import download_prices
from src.data.download_data import validate_download
from src.data.download_data import remove_empty_tickers

In [2]:
# ============================
# DATA EXTRACTION CONFIGURATION
# ============================

START_DATE = "2010-01-01"
END_DATE = "2024-12-31"
INTERVAL = "1d"
AUTO_ADJUST = False

In [3]:
# ====================================
# S&P 500 CONSTITUENTS
# ====================================

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500 = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})[0]

# Yahoo Finance uses "-" instead of "."
tickers = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

print(f"Number of constituents: {len(tickers)}")

Number of constituents: 503


Vemos que en principio tenemos 503 tickers potenciales aunque luego dependiendo de si estos tickers responden a empresas verdaderas o a cotizaciones no nulas pueden ser menos. 

In [4]:
prices = download_prices(
    tickers=tickers,
    START_DATE=START_DATE,
    END_DATE=END_DATE,
    INTERVAL=INTERVAL,
    AUTO_ADJUST=AUTO_ADJUST,
)

report = validate_download(prices, tickers)


[                       0%                       ]  2 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[****************      33%                       ]  166 of 503 completed$FDXF: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[**********************48%                       ]  242 of 503 completed$Q: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[**********************67%*******                ]  336 of 503 completed$HONA: possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1262322000, endDate = 1735621200")
[*********************100%***********************]  503 of 503 co

DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 499
Missing tickers   : 0
Empty tickers (NaN): 4

Empty tickers (100% NaN values):
  - FDXF
  - HONA
  - Q
  - SNDK


tenemos cuatro tickers que presentan cero valores distintos de NaN. Al reestructurar ahora el formato como veremos a continuación esos cuatro activos pasan de ser 4 tickers sin valores dentro a "eliminarse" (pasarán a la categoría de "missing tickers"). 

In [5]:

prices = remove_empty_tickers(prices)

# for datetime index in the correct format, we convert the index to datetime
prices.index = pd.to_datetime(prices.index)

# tranform name in string to avoid problems with MultiIndex
prices.columns = pd.MultiIndex.from_tuples(
    [(str(c[0]), str(c[1])) for c in prices.columns],
    names=["Price", "Ticker"]
)

# report for validation of the downloaded data
report = validate_download(prices, tickers)


DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 499
Missing tickers   : 4
Empty tickers (NaN): 0

Missing tickers (not returned by API):
  - FDXF
  - HONA
  - Q
  - SNDK


De los 503 tickers resulta que solo hemos podido descargar datos para 499 empresas 

In [6]:
# upload the S&P 500 constituents and prices to CSV files

prices.to_parquet("../data/raw/sp500_prices.parquet")

sp500.to_csv("../data/raw/sp500_constituents.csv", index=False)